In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U langchain-community
!pip install faiss-cpu
!pip install rank_bm25
!pip install transformers==4.44.2
!pip install -U FlagEmbedding

In [ ]:
import os
import json
import numpy as np
import torch
from tqdm import tqdm

from typing import List
from transformers import AutoTokenizer, AutoModel
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from tqdm import tqdm
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from collections import defaultdict
from langchain.load import dumps, loads
from FlagEmbedding import FlagReranker

from openai import OpenAI
from openai import AzureOpenAI

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

openai_key=os.getenv("OPENAI_API_KEY")
nebius_key=os.getenv("NEBIUS_API_KEY")
azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
nebius_base_url=os.getenv("NEBIUS_BASE_URL")

**1. Эмбеддинг**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
embeddings_folder = "/content/drive/MyDrive/Colab Notebooks/embedded_chunks"

In [ ]:
class DeepVKEmbeddings(Embeddings):
    def __init__(self, model_name: str = "deepvk/USER-base", device: str = "cuda", batch_size: int = 16):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.device = device
        self.batch_size = batch_size
        self.model.to(device)
        self.model.eval()

    @staticmethod
    def mean_pooling(model_output, attention_mask):
        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        all_embeddings = []
        for i in tqdm(range(0, len(texts), self.batch_size), desc="Векторизация"):
            batch = texts[i:i + self.batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=512)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
                embeddings = self.mean_pooling(outputs, inputs["attention_mask"]).cpu().numpy()
                all_embeddings.extend(embeddings)
        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

In [ ]:
embeddings = np.load(os.path.join(embeddings_folder, "embeddings.npy"))
with open(os.path.join(embeddings_folder, "metadata.json"), "r", encoding="utf-8") as f:
    metadata = json.load(f)
with open(os.path.join(embeddings_folder, "texts.json"), "r", encoding="utf-8") as f:
    texts = json.load(f)

**2. Индексация и ретривер**

In [ ]:
embedding_model = DeepVKEmbeddings(device=device)

In [ ]:
faiss_vectorstore = FAISS.load_local(
    "/content/drive/MyDrive/Colab Notebooks/faiss",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

In [ ]:
documents = [
    Document(
        page_content=text,
        metadata={**meta, "document_id": i}
    )
    for i, (text, meta) in enumerate(zip(texts, metadata))
]

**BM25**

<img src="bm25.png" style="width:100%;" />

In [ ]:
bm25_retriever = BM25Retriever.from_documents(documents)

**3. Reciprocal Rank Fusion**

<img src="rrf.png" style="width:70%;" />

In [ ]:
def reciprocal_rank_fusion(results: list[list], k=60):
    fused_scores = defaultdict(float)
    for docs in results:
        for rank, doc in enumerate(docs):
            doc_str = dumps(doc)
            fused_scores[doc_str] += 1 / (rank + k)
    reranked_results = [
        (loads(doc_str), score)
        for doc_str, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    return reranked_results

**4. Сross Encoder**

Использована модель [BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3)

In [ ]:
reranker = FlagReranker('BAAI/bge-reranker-v2-m3', use_fp16=True)

**5. Формирование json-структуры**

In [ ]:
def serialize_chunks(docs, scores, text_len=1800):
    items = []
    for doc, score in zip(docs, scores):
        items.append({
            "mnn": doc.metadata.get("mnn"),
            "trade_name": doc.metadata.get("trade_name"),
            "drug_id": doc.metadata.get("drug_id"),
            "file_name": doc.metadata.get("file_name"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "document_id": doc.metadata.get("document_id"),
            "score": score,
            "text": doc.page_content.strip().replace("\n", " ")[:text_len]
        })
    return items

**6. Генерация вопросов**

In [ ]:
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 15})
bm25_retriever.k = 15

ensemble_retriever_question = EnsembleRetriever(
    retrievers=[
        faiss_retriever,
        bm25_retriever
    ]
)

In [ ]:
client = AzureOpenAI(
    openai_api_key = os.getenv("OPENAI_API_KEY"),
    openai_api_version = "2024-12-01-preview",
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
)
deployment_name = 'gpt-4o-3',
temperature = 0.5

In [ ]:
def generate_questions(
    mnn: str,
    trade_names: list[str],
    retriever_question=ensemble_retriever_question,
    client=client,
    deployment="gpt-4o-3",
    top_k: int = 10
) -> dict:

    query = f"{mnn} " + " ".join(trade_names)

    # ensemble_retriever
    faiss_docs = faiss_retriever.get_relevant_documents(query)
    bm25_docs = bm25_retriever.get_relevant_documents(query)

    # RRF
    rrf_results = reciprocal_rank_fusion([faiss_docs, bm25_docs])
    docs = [doc for doc, _ in rrf_results]

    # CrossEncoder
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.compute_score(pairs, normalize=True)

    # сортировка и удаление дубликатов
    scored_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    seen = set()
    unique_scored_docs = []
    for doc, score in scored_docs:
        key = (doc.metadata.get("chunk_id"), doc.metadata.get("file_name"))
        if key not in seen:
            seen.add(key)
            unique_scored_docs.append((doc, score))
        if len(unique_scored_docs) == top_k:
            break

    if unique_scored_docs:
        reranked_docs, sorted_scores = zip(*unique_scored_docs)
    else:
        reranked_docs, sorted_scores = [], []

    context = "\n\n".join([doc.page_content for doc in reranked_docs])

    # генерируем trade_name_questions динамически
    trade_template = ", ".join(
    f'"{tn}": []'
    for tn in trade_names
    )

    prompt = f"""
    Ты — пациент, изучающий инструкцию к препарату. У тебя есть действующее вещество: "{mnn}" и торговые названия: {', '.join(trade_names)}.
    В инструкции описаны разделы: показания, противопоказания, побочные эффекты, дозировка, способ применения и взаимодействие с другими лекарствами.

    Инструкция:
    {context}

    На основе текста инструкции сформулируй от лица пациента:
    - 3 разнообразных вопроса про действующее вещество ({mnn})
    - 2 разнообразных вопроса про каждое торговое название

    При этом включи вопросы следующих типов:
    – Открытый (например, “Опишите…”)
    – Да/нет (например, “Можно ли…?”)
    – Ситуационный (например, “Если у меня…?”)
    – Про дозировку
    – Про противопоказания
    – Про побочные эффекты
    – Про способ применения
    – Про взаимодействие с другими лекарствами

    Верни только чистый JSON в формате:
    {{
      "mnn_questions": ["…","…","…"],
      "trade_name_questions": {{{trade_template}}}
    }}
    """.strip()

    try:
        messages = [
          {"role":"system", "content":
            "Ты — ассистент по созданию вопросов к инструкциям."
            "Генерируй разнообразные и релевантные вопросы, основываясь на тексте инструкции."
          },
          {"role":"user", "content": prompt}
        ]
        resp = client.chat.completions.create(
            model=deployment,
            messages=messages,
            max_tokens=800,
            temperature=0.5,
            response_format={"type": "json_object"}
        )

    except Exception as e:
        print("Ошибка подключения к Azure (questions):", e)
        raise

    # если SDK вернул dict, то берём его сразу, иначе парсим
    if isinstance(resp, dict):
        q_json = resp
    else:
        raw = resp.choices[0].message.content.strip()
        print("RAW QUESTIONS RESPONSE:\n", raw)
        q_json = json.loads(raw)

    # метаданные чанков
    q_json["question_chunks"] = serialize_chunks(reranked_docs, sorted_scores)
    return q_json

**7. Генерация ответов**

In [ ]:
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 15})
bm25_retriever.k = 15

ensemble_retriever_answer = EnsembleRetriever(
    retrievers=[
        faiss_retriever,
        bm25_retriever,
    ]
)

In [ ]:
client = OpenAI(
    base_url=os.getenv("NEBIUS_BASE_URL"),
    nebius_api_key=os.getenv("NEBIUS_API_KEY"),
)

In [ ]:
def generate_answer(
    question: str,
    mnn: str,
    trade_names: list[str],
    retriever_answer=ensemble_retriever_answer,
    client=client,
    model_id="mistralai/Mistral-Nemo-Instruct-2407",
    top_k: int = 10
):
    # ensemble_retriever
    faiss_docs = faiss_retriever.get_relevant_documents(question)
    bm25_docs = bm25_retriever.get_relevant_documents(question)

    # RRF
    rrf_results = reciprocal_rank_fusion([faiss_docs, bm25_docs])
    docs = [doc for doc, _ in rrf_results]

    # CrossEncoder
    pairs = [[question, doc.page_content] for doc in docs]
    scores = reranker.compute_score(pairs, normalize=True)

    # сортировка по убыванию score и удаление дубликатов
    scored_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    seen = set()
    unique_scored_docs = []
    for doc, score in scored_docs:
        key = (doc.metadata.get("chunk_id"), doc.metadata.get("file_name"))
        if key not in seen:
            seen.add(key)
            unique_scored_docs.append((doc, score))
        if len(unique_scored_docs) == top_k:
            break

    # вдруг уникальных чанков меньше, чем top_k
    if unique_scored_docs:
        reranked_docs, sorted_scores = zip(*unique_scored_docs)
    else:
        reranked_docs, sorted_scores = [], []

    context = "\n\n".join([doc.page_content for doc in reranked_docs])

    prompt = f"""
    Ты — медицинский консультант. Отвечай строго на русском языке.
    Используй только текст инструкции для предоставления ответа.
    Дай чёткий и краткий ответ на вопрос пациента.
    Если в инструкции нет ответа, то посоветуй обратиться к врачу.

    Инструкция:
    {context}

    Вопрос пациента:
    {question}

    Верни ответ в следующем JSON-формате:
    {{
      "answer": "..."
    }}

    """.strip()

    try:
        messages = [
          {"role":"system", "content":
            "Ты — медицинский консультант и отвечаешь на вопросы пациентов."
            "Давай ответы только на русском языке на основе текста инструкций."
          },
          {"role":"user", "content": prompt}
        ]

        resp = client.chat.completions.create(
            model=model_id,
            messages=messages,
            max_tokens=800,
            temperature=0.2,
            response_format={"type": "json_object"}
        )
    except Exception as e:
        print("Ошибка подключения к OpenAI (answers):", e)
        raise

    # если SDK вернул dict, то берём его сразу, иначе парсим
    if isinstance(resp, dict):
        a_json = resp
    else:
        raw = resp.choices[0].message.content.strip()
        print("RAW ANSWER RESPONSE:\n", raw)
        a_json = json.loads(raw)

    # метаданные чанков
    a_json["answer_chunks"] = serialize_chunks(reranked_docs, sorted_scores)
    return a_json

**8. Автоматическая генерарция пар вопросов и ответов**

In [ ]:
def run_qa_pipeline(
    mnn: str,
    trade_names: list[str],
    outfile: str | None = None
) -> dict:
    result = {
        "mnn_questions": [],
        "trade_name_questions": {tn: [] for tn in trade_names}
    }

    q_block = generate_questions(mnn, trade_names)

    for q in q_block["mnn_questions"]:
        a_block = generate_answer(q, mnn, trade_names)
        result["mnn_questions"].append({
            "question": q,
            "question_chunks": q_block["question_chunks"],
            "answer": a_block["answer"],
            "answer_chunks": a_block["answer_chunks"]
        })

    for tn, qs in q_block["trade_name_questions"].items():
        for q in qs:
            a_block = generate_answer(q, mnn, [tn])
            result["trade_name_questions"][tn].append({
                "question": q,
                "question_chunks": q_block["question_chunks"],
                "answer": a_block["answer"],
                "answer_chunks": a_block["answer_chunks"]
            })

    if outfile:
        with open(outfile, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print("Сохранено:", outfile)

    return result

**9. Генерация тестового датасета**


In [ ]:
output_path = "/content/drive/MyDrive/Colab Notebooks/test_dataset_28_mnn.json"

selected_mnn = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/selected_mnn.csv", dtype=str)["mnn"].tolist() # отобранные 28 МНН

meta = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/instructions_metadata.csv", dtype=str)

test_dataset = {}
for mnn in tqdm(selected_mnn, desc="Генерация тестового датасета", unit="MNN"):
    trade_names = meta.loc[meta["mnn"] == mnn, "trade_name"].tolist()
    test_dataset[mnn] = run_qa_pipeline(mnn, trade_names)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_dataset, f, ensure_ascii=False, indent=2)

**Подсчёт количества QA-пар**

In [ ]:
with open(output_path, "r", encoding="utf-8") as f:
    data = json.load(f)

total_pairs = 0
mnn_question_pairs = 0
trade_name_question_pairs = 0

for mnn_block in data.values():
    # вопросы по МНН
    if "mnn_questions" in mnn_block:
        if isinstance(mnn_block["mnn_questions"], list):
            mnn_q_count = len(mnn_block["mnn_questions"])
            mnn_question_pairs += mnn_q_count
            total_pairs += mnn_q_count

    # вопросы по торговым названиям
    if "trade_name_questions" in mnn_block:
        if isinstance(mnn_block["trade_name_questions"], dict):
            for qlist in mnn_block["trade_name_questions"].values():
                tn_q_count = len(qlist)
                trade_name_question_pairs += tn_q_count
                total_pairs += tn_q_count

print(f"Количество пар вопрос–ответ: {total_pairs}")
print(f"Из них вопросов по МНН: {mnn_question_pairs}")
print(f"Из них вопросов по торговым названиям: {trade_name_question_pairs}")

Количество пар вопрос–ответ: 280
Из них вопросов по МНН: 84
Из них вопросов по торговым названиям: 196
